In [ ]:
import sys
from pathlib import Path
import os

# Get the directory where this notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent  # Go up one level to project root
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
# import matplotlib.pyplot as plt
from cs_priors.simulation.mixing_model import run_simulation
from cs_priors.plotting.plot_complex import plot_matrices
from cs_priors.plotting.plotting import plot_overview
import cs_priors.solvers.sparse_prior as sp
import cs_priors.solvers.vectorized_sparse_prior as vsp


# Check that the two methods are equivalent

In [ ]:
sim = run_simulation(num_mics=3, num_sources=5, s_sparse=2)
A = sim.C[:,:, 1]
Y = sim.Y[:, 1]
X_TRUE = sim.X[:, 1]
X0 = np.linalg.pinv(A) @ Y.reshape(-1,1)

plot_overview(sim)

# Comparing the objective functions
We initalize random $z$ values and expect the functions to compute the same objective and the same gradient of the objective

### Initial values

In [ ]:
# Helper functions
def covariance_matrices(
    num_sources: int, variance: float = 1.0, spread: float = 0.005
    ) -> list[np.ndarray]:
    return [
        np.diag([variance if j == i else spread for j in range(num_sources)])
        for i in range(num_sources)
    ]

# Quadratic form x^T D x for each grid point
def quad_form(points: np.ndarray, D_matrix: np.ndarray) -> float:
    # This function handles both a single column vector (shape N, 1)
    # and a batch of row vectors (shape ..., N)
    if points.shape[-1] == 1 and points.ndim > 1:
        # It's a single column vector, use the mathematical formula
        return (points.T @ D_matrix @ points).item()
    else:
        # It's a batch of row vectors, use the efficient NumPy way
        return (points @ D_matrix * points).sum(axis=-1)

X0_real = sp.to_real_augmented(X0)

U, S, Vh = np.linalg.svd(A)
rank = np.sum(S > 1e-10)
B = Vh[rank:].conj().T  # Shape (num_sources, dim_null_space)
B_real = sp.to_real_augmented(B)
variance = 1.0
spread = 0.005
delta = variance - spread
D = covariance_matrices(num_sources=B_real.shape[0], variance=variance, spread=spread)
z = np.random.randn(B_real.shape[1]) * 10

plot_matrices([Y, A, X_TRUE, X0, X0_real,B_real, z], titles=['Y', 'A', 'X_TRUE', 'X0', 'X0_real','B_real', 'z'])

### Comparing objective functions for the same value

In [ ]:
def vec_negative_objective(z: np.ndarray) -> float:
    x = (X0_real.reshape(-1, 1) + B_real @ z.reshape(-1, 1)).flatten()

    norm_squared = np.sum(x**2) 
    # sum_i exp(-x^T D_i x) = exp(-spread * ||x||^2) * sum_i exp(-delta * x[i]^2)
    global_factor = np.exp(-spread * norm_squared)
    individual_sum = np.sum(np.exp(-delta * x**2))

    return -global_factor * individual_sum

def sp_negative_objective(z: np.ndarray) -> float:
    # numpy broadcasting will make row vectors if z is 1D
    # X0_real has shape (N,) whereas B_real @ z has shape (N, 1)
    x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1)).reshape(-1, 1)
    return -sum(np.exp(-quad_form(x, D_i)) for D_i in D)
    

vec_obj = vec_negative_objective(z)
sp_obj = sp_negative_objective(z)

print(f"Vectorized Sparse Prior Objective: {vec_obj}")
print(f"Sparse Prior Objective: {sp_obj}")
assert np.isclose(vec_obj, sp_obj), "Objectives do not match!"


### Comparing derivatives for the same values

In [ ]:
def vec_first_derivative_objective(z: np.ndarray) -> np.ndarray:
        # (-1) e^{-(x_0 + Bz)^T D (x_0+Bz)}(B^T(D + D^T) (x_0 + B z))
        # where D_i is symmetric
        # It can be computed more efficiently as follows:
        # B^T (-2e^{-(-spread * ||x||^2} * [ spread * x * sum_i e^{-delta * x[i]^2} + delta * elementwise(x, e^{-delta * x^2}) ]
        # where delta = variance - spread
        x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
        x_flat = x.flatten()

        # Compute quadratic forms efficiently
        norm_squared = np.sum(x_flat**2)
        global_factor = np.exp(-spread * norm_squared)
        exp_terms = np.exp(-delta * x**2)
        individual_sum = np.sum(exp_terms)

        dx = -2 * global_factor * (spread * x * individual_sum + delta * x * exp_terms)
        grad = B_real.T @ dx

        return -grad.reshape(-1)

def sp_first_derivative_objective(z: np.ndarray) -> np.ndarray:
        # (-1) e^{-(x_0 + Bz)^T D (x_0+Bz)}(B^T(D + D^T) (x_0 + B z))
        # where D_i is symmetric
        x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
        grad = np.zeros_like(z)
        for D_i in D:
            qf = quad_form(x, D_i)
            exp_neg_qf = np.exp(-qf)
            grad += 2 * exp_neg_qf * (B_real.T @ (D_i @ x)).flatten()
        return grad  # this is the negative, not actual

vec_derivative = vec_first_derivative_objective(z)
sp_derivative = sp_first_derivative_objective(z)

print(f"Vectorized Sparse Prior Objective: {vec_derivative}")
print(f"Sparse Prior Objective: {sp_derivative}")
assert np.isclose(vec_derivative, sp_derivative).all(), "Derivatives do not match!"

### Comparing the full functions

In [ ]:
import scipy.optimize as optimize
from cs_priors.solvers.sparse_prior import from_real_augmented

# 1. OG SP
def sp_optimize_objective(X0_real, B_real, D, callback=None, z_start=None):
    # factory function for the objective to minimize
    def negative_objective(z: np.ndarray) -> float:
        # numpy broadcasting will make row vectors if z is 1D
        # X0_real has shape (N,) whereas B_real @ z has shape (N, 1)
        x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1)).reshape(-1, 1)
        return -sum(np.exp(-quad_form(x, D_i)) for D_i in D) 

    def first_derivative_objective(z: np.ndarray) -> np.ndarray:
        # (-1) e^{-(x_0 + Bz)^T D (x_0+Bz)}(B^T(D + D^T) (x_0 + B z))
        # where D_i is symmetric
        x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
        grad = np.zeros_like(z)
        for D_i in D:
            qf = quad_form(x, D_i)
            exp_neg_qf = np.exp(-qf)
            grad += 2 * exp_neg_qf * (B_real.T @ (D_i @ x)).flatten()
        return grad  # this is the negative, not actual

    def second_derivative_objective(z: np.ndarray) -> np.ndarray:
        # sum_i e^{-(x_0 + Bz)^T D_i (x_0+Bz)}
        #  * [ (B^T (D_i + D_i^T) (x_0 + B z)) (B^T (D_i + D_i^T) (x_0 + B z))^T
        #      - B^T (D_i + D_i^T) B ]
        x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
        hess = np.zeros((z.shape[0], z.shape[0]))
        for D_i in D:
            qf = quad_form(x, D_i)
            exp_neg_qf = np.exp(-qf)
            BdX = B_real.T @ (D_i @ x)
            hess += exp_neg_qf * (
                2 * np.outer(BdX, BdX) - 2 * (B_real.T @ D_i @ B_real)
            )
        return -hess  # this is the negative Hessian, not the actual

    if z_start is None:
        z_start = np.zeros(B_real.shape[1])

    res = optimize.minimize(
        negative_objective,
        z_start,
        method='l-BFGS-B',
        callback=callback,
        jac=first_derivative_objective,
        # hess=second_derivative_objective,
    )
    # print(res.message)
    z_opt = res.x
    x_opt = X0_real.reshape(-1, 1) + (B_real @ z_opt.reshape(-1, 1))
    x_opt_complex = from_real_augmented(x_opt)
    return z_opt, x_opt, x_opt_complex, res

# 2. Vectorized SP
def vec_optimize_real_valued_objective(
    X0_real,
    B_real,
    variance: float = 1.0,
    spread: float = 0.005,
    callback=None,
    z_start=None,
):
    delta = variance - spread  # Coefficient difference

    def negative_objective(z: np.ndarray) -> float:
        x = (X0_real.reshape(-1, 1) + B_real @ z.reshape(-1, 1)).flatten()

        norm_squared = np.sum(x**2)
        # sum_i exp(-x^T D_i x) = exp(-spread * ||x||^2) * sum_i exp(-delta * x[i]^2)
        global_factor = np.exp(-spread * norm_squared)
        individual_sum = np.sum(np.exp(-delta * x**2))

        return -global_factor * individual_sum

    def first_derivative_objective(z: np.ndarray) -> np.ndarray:
        # (-1) e^{-(x_0 + Bz)^T D (x_0+Bz)}(B^T(D + D^T) (x_0 + B z))
        # where D_i is symmetric
        # It can be computed more efficiently as follows:
        # B^T (-2e^{-(-spread * ||x||^2} * [ spread * x * sum_i e^{-delta * x[i]^2} + delta * elementwise(x, e^{-delta * x^2}) ]
        # where delta = variance - spread
        x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
        x_flat = x.flatten()

        # Compute quadratic forms efficiently
        norm_squared = np.sum(x_flat**2)
        global_factor = np.exp(-spread * norm_squared)
        exp_terms = np.exp(-delta * x**2)
        individual_sum = np.sum(exp_terms)

        dx = -2 * global_factor * (spread * x * individual_sum + delta * x * exp_terms)
        grad = B_real.T @ dx

        return -grad.reshape(-1)

    if z_start is None:
        z_start = np.zeros(B_real.shape[1], dtype=float)

    result = optimize.minimize(
        negative_objective,
        z_start,
        jac=first_derivative_objective,
        callback=callback,
        method="L-BFGS-B",
    )

    # print(res.message)
    z_opt = result.x
    x_opt = X0_real.reshape(-1, 1) + (B_real @ z_opt.reshape(-1, 1))
    x_opt_complex = from_real_augmented(x_opt)
    return z_opt, x_opt, x_opt_complex, result

_, _, vec_sol, _ = vec_optimize_real_valued_objective(X0_real=X0_real, B_real=B_real)
_, _, sp_sol, _ = sp_optimize_objective(X0_real=X0_real, B_real=B_real, D=D)

print(f'{vec_sol=}')
print(f'{sp_sol=}')
assert np.isclose(vec_sol, sp_sol).all(), "Solutions do not match"


# Compare full 

In [ ]:
X_sp, B = sp.sparse_prior_solution(X0,A)
X_vsp, B = vsp.sparse_prior_solution(X0,A)

assert np.isclose(X_sp, X_vsp).all(), "Final solutions do not match!"
plot_matrices([X_TRUE, X0, X_sp, X_vsp], titles=["True", "Moore-Penrose", "Sparse Prior", "Vectorized Sparse Prior"])

# Compare performance

In [ ]:
import time
from cs_priors.plotting.plotting import wrapper_plot_geometry
sim = run_simulation(num_mics=10, num_sources=100, s_sparse=10)
A = sim.C[:,:, 1]
Y = sim.Y[:, 1]
X_TRUE = sim.X[:, 1]
X0 = np.linalg.pinv(A) @ Y.reshape(-1,1)
wrapper_plot_geometry(sim)

In [ ]:
time_start = time.time()
X_sp, B = sp.sparse_prior_solution(X0,A)
time_sp = time.time() - time_start
print(f"Sparse Prior time: {time_sp:.4f} seconds")

time_start = time.time()
X_vsp, B = vsp.sparse_prior_solution(X0,A)
time_vsp = time.time() - time_start
print(f"Vectorized Sparse Prior time: {time_vsp:.4f} seconds")

assert np.isclose(X_sp, X_vsp).all(), "Final solutions do not match!"

### Compare time with LASSO

In [ ]:
from cs_priors.solvers.complex_lasso import complex_lasso

time_start = time.time()
X_lasso = complex_lasso(A, Y, alpha=1e-4)
time_lasso = time.time() - time_start
print(f"Complex LASSO time: {time_lasso:.4f} seconds")

### Quantify speedup

In [ ]:
import random

# TODO: Figure out why this is insufficient to ensure reproducibility
random.seed(0)
np.random.seed(0)

num_seeds = 100 #100

sp_times = np.zeros(num_seeds)
vsp_times = np.zeros(num_seeds)
lasso_times = np.zeros(num_seeds)

for seed in range(num_seeds):
    # random.choice affects which sources are active
    random.seed(seed)
    np.random.seed(seed)
    
    sim = run_simulation(num_mics=10, num_sources=100, s_sparse=10)
    A = sim.C[:,:, 1]
    Y = sim.Y[:, 1]
    X_TRUE = sim.X[:, 1]
    X0 = np.linalg.pinv(A) @ Y.reshape(-1,1)

    time_start = time.time()
    X_sp, B = sp.sparse_prior_solution(X0,A)
    sp_times[seed] = time.time() - time_start

    time_start = time.time()
    X_vsp, B = vsp.sparse_prior_solution(X0,A)
    vsp_times[seed] = time.time() - time_start

    time_start = time.time()
    X_lasso = complex_lasso(A, Y, alpha=1e-4)
    lasso_times[seed] = time.time() - time_start

# Color codes
GREEN = '\033[92m'
BOLD = '\033[1m'
RESET = '\033[0m'

print(f"Average Sparse Prior time: {np.mean(sp_times):.4f} seconds")
print(f"Average Vectorized Sparse Prior time: {np.mean(vsp_times):.4f} seconds")
print(f"Average Complex LASSO time: {np.mean(lasso_times):.4f} seconds")

print()
print(f"Complex LASSO faster than Sparse Prior in {np.sum(lasso_times < sp_times)} out of {num_seeds} runs")
print(f"Vectorized Sparse Prior faster than Sparse Prior in {np.sum(vsp_times < sp_times)} out of {num_seeds} runs")
print(f"Vectorized Sparse Prior faster than Complex LASSO in {GREEN}{BOLD}{np.sum(vsp_times < lasso_times)}{RESET} out of {num_seeds} runs")

print()
print(f"On average, Vectorized Sparse Prior is {np.mean(sp_times / vsp_times):.2f} times faster than Sparse Prior")
print(f"On average, Complex LASSO is {np.mean(sp_times / lasso_times):.2f} times faster than Sparse Prior")
print(f"On average, Vectorized Sparse Prior is {GREEN}{BOLD}{np.mean(lasso_times / vsp_times):.2f}{RESET} times faster than Complex LASSO")
print(f"On average, Complex LASSO is {np.mean(vsp_times / lasso_times):.2f} times faster than Vectorized Sparse Prior")


# Compare average error

In [ ]:
np.random.seed(0)
random.seed(0) # running cells in a notebook will alter the following random state
sources = 100
mic_range = [int(p * sources) for p in [0.1, 0.6, 0.8, 1.0]]
sparsity_range = [1,2,3,4,5]
seeds = 10
debug = False
alpha = 1e-3

sim = run_simulation(num_mics=mic_range[0], num_sources=sources, s_sparse=sparsity_range[0], angle_step=2*np.pi/sources)
wrapper_plot_geometry(sim, skip_labels=True)


In [ ]:
from spaghetti_code.old_spaghetti.compare_sparsity_LASSO import heatmap_average
heatmap_average(mic_range=mic_range, sources=sources, sparsity_range=sparsity_range, num_seeds=seeds, alpha=alpha, debug=debug)

# Group Lasso

In [ ]:
from cs_priors.plotting.plotting import wrapper_plot_geometry
import random

num_sources = 10
num_mics = int(num_sources * 0.8)
s_sparse = 1


random.seed(0)
sim = run_simulation(num_mics=num_mics, num_sources=num_sources, s_sparse=s_sparse, angle_step=2*np.pi/num_sources, distance=1.5)
A = sim.C[:,:, 1]
Y = sim.Y[:, 1]
X_TRUE = sim.X[:, 1]
X0 = np.linalg.pinv(A) @ Y.reshape(-1,1)

wrapper_plot_geometry(sim)

In [ ]:
from cs_priors.solvers.c_lasso import c_lasso
from cs_priors.solvers.complex_lasso import complex_lasso



X_c_lasso = c_lasso(Y, A, alpha=0.001, max_iter=100000)
X_lasso = complex_lasso(A, Y, alpha=0.001)
X_sp = vsp.sparse_prior_solution(X0, A)[0]

plot_matrices([X_TRUE, X0, X_c_lasso, X_lasso, X_sp], titles=["True", "Moore-Penrose", "diy C-LASSO", "Aug. real form LASSO", "Vectorized Sparse Prior "])

In [ ]:
import cs_priors.solvers.c_vec_sp as cvsp
X_cvsp = cvsp.sparse_prior_solution(X0, A)[0]

plot_matrices([X_TRUE, X0, X_sp, X_cvsp], titles=["True", "Moore-Penrose", "Vectorized Sparse Prior", "C-Vec-SP"])